## Replication: The Fake News Effect (Thaler, 2024)

**Journal**: AEJ: Microeconomics, Vol. 16, No. 2 (May 2024)
**DOI**: https://doi.org/10.1257/mic.20220146

**Design summary**: Subjects guess medians on politicized factual questions.
They receive a binary True News / Fake News message (uninformative to a Bayesian)
and report their belief that the message came from a true source.
Motivated reasoning predicts Pro-Party news gets higher veracity assessments.

**Replication target**: Reproduce Figures 1-4 and Tables 2-3 using Python.
Original code is in Stata (.do files). All code below is original Python.

### Environment Setup
If you are running this notebook in Binder, no setup is needed. Binder automatically installs the packages listed in `requirements.txt` when it builds the environment.

If you are running this notebook locally on your own machine, install the required packages before running the notebook:

```bash
pip install -r requirements.txt
```

### Library Imports
Import standard scientific Python stack plus statsmodels for panel OLS
and linearmodels for fixed-effects regression with clustered SEs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from scipy.special import logit

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'serif',
    'axes.grid': False
})

### Data Loading
Load the pre-cleaned dataset from the AEA replication archive.
Use either the upload button below or keep `cleaned_data.csv` in the repository's `data/` folder.
The notebook uses the cleaned dataset because it includes derived variables such as
`pro_party`, `anti_party`, `partisan`, and `prob_true_demeaned`.


In [ ]:
from pathlib import Path
from io import BytesIO
from IPython.display import display

DATA_PATH = Path('data/cleaned_data.csv')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)


def load_data(source):
    """Load and return the cleaned experiment dataset from a path or uploaded bytes."""
    return pd.read_csv(source, low_memory=False)


def _uploaded_file_bytes(upload_widget):
    """Return uploaded file bytes for both ipywidgets 7 and 8 formats."""
    if upload_widget is None:
        return None
    value = upload_widget.value
    if isinstance(value, dict):
        if not value:
            return None
        return next(iter(value.values()))['content']
    if isinstance(value, tuple):
        if not value:
            return None
        return value[0]['content']
    return None


try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    uploader = None
    print('Optional upload widget unavailable. Run `pip install -r requirements.txt`,')
    print('or place cleaned_data.csv in the data/ folder before running the next cell.')
else:
    uploader = widgets.FileUpload(accept='.csv', multiple=False)
    display(uploader)
    print('Upload cleaned_data.csv with the button above, then run the next cell.')
    print('Binder/GitHub option: keep cleaned_data.csv in data/ and skip the upload button.')


In [ ]:
uploaded_bytes = _uploaded_file_bytes(uploader)

if uploaded_bytes is not None:
    df = load_data(BytesIO(uploaded_bytes))
elif DATA_PATH.exists():
    df = load_data(DATA_PATH)
else:
    raise FileNotFoundError(
        'Upload cleaned_data.csv with the file picker, or place cleaned_data.csv '
        'in the data/ folder before running this cell.'
    )

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")


### Sample Construction
Apply the three main sample filters from the paper:
1. Attention-check passers only (all rows in cleaned_data.csv already passed)
2. Non-neutral subjects: net_party != 0 (Pro-Dem or Pro-Rep)
3. For news assessment regressions: must have received a news message
   (obs where message_greater==1 or message_less==1)
N = 987 unique subjects (non-neutral) generating 7,902 politicized news obs.


In [ ]:
def build_analysis_samples(df):
    """Return dict of key analysis subsets used throughout the paper."""
    has_news = (df['message_greater'] + df['message_less']) >= 1
    non_neutral = df['net_party'] != 0
    politicized = df['politicized_news'] == 1
    party_news = (df['pro_party'] == 1) | (df['anti_party'] == 1)
    return {
        'main': df[non_neutral & has_news & party_news].copy(),
        'politicized': df[non_neutral & has_news & politicized].copy(),
        'all_topics': df[non_neutral & has_news].copy(),
        'second_guess': df[non_neutral & (df['your_reguess'].notna())].copy()
    }

samples = build_analysis_samples(df)
print("Main sample (Pro/Anti-Party, non-neutral):", len(samples['main']))
print("Politicized topics sample:", len(samples['politicized']))


### Table 4: Prior Beliefs by Party
Compare initial median guesses (your_answer) between Pro-Rep and Pro-Dem subjects
on each politicized topic. The sign of the difference should match Table 1's
hypothesized motive directions (e.g., Pro-Rep guesses Obama crime rate was higher).

In [ ]:
def winsorize_col(s, pct=0.05):
    """Winsorize series at pct and 1-pct quantiles."""
    lo, hi = s.quantile(pct), s.quantile(1 - pct)
    return s.clip(lo, hi)

def build_prior_beliefs_table(df):
    """Replicate Table 4: prior beliefs by party for politicized topics."""
    topics = df[df['politicized_news'] == 1].copy()
    topics = topics[topics['net_party'] != 0]
    topics['ans_w'] = topics.groupby('topic')['your_answer'].transform(winsorize_col)
    rep_mask = topics['pro_rep'] == 1
    dem_mask = topics['pro_dem'] == 1
    return topics.groupby('topic').apply(
        lambda g: pd.Series({
            'Pro-Rep Mean': g.loc[g['pro_rep']==1, 'ans_w'].mean(),
            'Pro-Dem Mean': g.loc[g['pro_dem']==1, 'ans_w'].mean(),
            'Difference': g.loc[g['pro_rep']==1,'ans_w'].mean() - g.loc[g['pro_dem']==1,'ans_w'].mean()
        })
    ).round(3)

prior_table = build_prior_beliefs_table(df)
print(prior_table)

### Table 5: Balance Check (Pro-Party vs Anti-Party Treatment)
Verify that demographic characteristics are balanced across the Pro-Party
and Anti-Party treatment arms. None should differ significantly (p > 0.05).


In [ ]:
def balance_test(df, covariates):
    """Run t-tests comparing pro_party=1 vs anti_party=1 on covariates."""
    pro = df[df['pro_party'] == 1]
    anti = df[df['anti_party'] == 1]
    rows = []
    for col in covariates:
        t, p = stats.ttest_ind(pro[col].dropna(), anti[col].dropna())
        rows.append({'Variable': col,
                     'Pro-Party Mean': pro[col].mean(),
                     'Anti-Party Mean': anti[col].mean(),
                     'Diff': pro[col].mean() - anti[col].mean(),
                     'p-value': round(p, 3)})
    return pd.DataFrame(rows)

balance_cols = ['abs_net_party', 'male', 'age', 'edu',
                'log_inc', 'white', 'red_state', 'religious_group']
bal = balance_test(samples['main'], balance_cols)
print(bal.to_string(index=False))

### Figure 1: CDF of Assessments for Pro-Party and Anti-Party News
The x-axis is subjects' stated Pr(True News | news received).
The y-axis is the share of observations at or below that value.
A Bayesian would have identical CDFs. The gap between them = motivated reasoning.

In [ ]:
def compute_ecdf(series):
    """Return x and y arrays for the empirical CDF."""
    x = np.sort(series.dropna().values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def plot_figure1(df_main):
    """Replicate Figure 1: CDF of news veracity for Pro- vs Anti-Party news."""
    pro = df_main[df_main['pro_party'] == 1]['prob_true']
    anti = df_main[df_main['anti_party'] == 1]['prob_true']
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.step(*compute_ecdf(anti), where='post', color='purple',
            label='Anti-Party news', linewidth=1.5)
    ax.step(*compute_ecdf(pro), where='post', color='green',
            linestyle='dotted', label='Pro-Party news', linewidth=1.5)
    ax.set_xlabel('Belief About Pr(True News)')
    ax.set_ylabel('Share of responses')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'figure1_replication.png', dpi=150)
    plt.show()

plot_figure1(samples['main'])

### Figure 2: Motivated Reasoning by News Direction and Partisanship
Group observations into 5 bins. Compute mean and 95% CI of prob_true_demeaned.
partisan = 1 when abs_net_party is above the sample median.
Key result: partisan Pro-Party bar is highest; partisan Anti-Party bar is lowest.

In [ ]:
def assign_party_group(row):
    """Assign one of five groups for Figure 2."""
    if row['anti_party'] and row['partisan']: return 1
    if row['anti_party'] and row['moderate']: return 2
    if row['neutral_news']: return 3
    if row['pro_party'] and row['moderate']: return 4
    if row['pro_party'] and row['partisan']: return 5
    return np.nan

def plot_figure2(df_all):
    """Replicate Figure 2 bar chart with 95% CI error bars."""
    d = df_all[(df_all['net_party'] != 0) & df_all['prob_true_demeaned'].notna()].copy()
    d['grp'] = d.apply(assign_party_group, axis=1)
    g = d.groupby('grp')['prob_true_demeaned']
    means = g.mean(); sems = g.sem(); ns = g.count()
    ci = stats.t.ppf(0.975, ns - 1) * sems
    colors = ['#501050', '#8c508c', '#787878', '#3c8c50', '#005014']
    labels = [f'Anti-Party,\nPartisans\nn = {ns[1]:,}',
              f'Anti-Party,\nModerates\nn = {ns[2]:,}',
              f'Neutral,\nAll Subjects\nn = {ns[3]:,}',
              f'Pro-Party,\nModerates\nn = {ns[4]:,}',
              f'Pro-Party,\nPartisans\nn = {ns[5]:,}']
    fig, ax = plt.subplots(figsize=(9, 5))
    xs = np.arange(1, 6)
    ax.bar(xs, means, color=colors, width=0.75, zorder=2)
    ax.errorbar(xs, means, yerr=ci, fmt='none', color='black', capsize=4, zorder=3)
    ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel('Pr(True), Demeaned'); ax.set_ylim(-0.1, 0.1)
    ax.axhline(0, color='black', linewidth=0.8)
    fig.tight_layout(); fig.savefig(FIGURES_DIR / 'figure2_replication.png', dpi=150); plt.show()

plot_figure2(samples['all_topics'])

### Figure 3: Motivated Reasoning and Assessments of Fake News
6-bar chart grouping by (news direction) x (actual veracity).
On politicized topics: Fake News bars should exceed True News bars within each direction.
On Neutral topics: no such pattern expected (and the paper finds the opposite --
subjects slightly favor True News on neutral topics).

In [ ]:
def assign_veracity_group(row):
    """Assign one of 6 groups for Figure 3 (direction x veracity)."""
    if row['anti_party'] and row['fake_news']: return 1
    if row['anti_party'] and row['true_news']: return 2
    if row['neutral_news'] and row['fake_news']: return 3
    if row['neutral_news'] and row['true_news']: return 4
    if row['pro_party'] and row['fake_news']: return 5
    if row['pro_party'] and row['true_news']: return 6
    return np.nan

def plot_figure3(df_all):
    """Replicate Figure 3: 6-bar chart of demeaned assessments."""
    d = df_all[(df_all['net_party'] != 0) & df_all['prob_true_demeaned'].notna()].copy()
    d = d[~d['ego_news'].astype(bool)]
    d['grp'] = d.apply(assign_veracity_group, axis=1)
    g = d.groupby('grp')['prob_true_demeaned']
    means = g.mean(); sems = g.sem(); ns = g.count()
    ci = stats.t.ppf(0.975, ns - 1) * sems
    colors = ['#501050','#501050','#909090','#909090','#1a6e36','#1a6e36']
    fig, ax = plt.subplots(figsize=(10, 5))
    xs = np.arange(1, 7)
    ax.bar(xs, means, color=colors, width=0.7, zorder=2)
    ax.errorbar(xs, means, yerr=ci, fmt='none', color='black', capsize=4, zorder=3)
    ax.set_xticks(xs)
    ax.set_xticklabels([f'Anti-Party,\nFake\nn={ns.get(1,0):,}',f'Anti-Party,\nTrue\nn={ns.get(2,0):,}',
                        f'Neutral,\nFake\nn={ns.get(3,0):,}',f'Neutral,\nTrue\nn={ns.get(4,0):,}',
                        f'Pro-Party,\nFake\nn={ns.get(5,0):,}',f'Pro-Party,\nTrue\nn={ns.get(6,0):,}'], fontsize=8.5)
    ax.set_ylabel('Pr(True), Demeaned'); ax.set_ylim(-0.1, 0.1)
    ax.axhline(0, color='black', linewidth=0.8)
    fig.tight_layout(); fig.savefig(FIGURES_DIR / 'figure3_replication.png', dpi=150); plt.show()

plot_figure3(samples['all_topics'])

### Figure 4: Motivated Reasoning by Topic (Coefficient Plot)
Run OLS of prob_true on pro_party x topic_dummy interactions,
with FE for subject, round, and topic. Cluster SEs at subject level.
The coefficient on each interaction is the pro_party effect for that topic.
8 of 9 politicized topics should be significant at p < 0.001.

In [ ]:
def fit_topic_model(df):
    d = df.copy()
    d['pro_party'] = d['pro_party'].astype(float)
    d['topic_id'] = d['topic_id'].astype(int)
    m = smf.ols('prob_true ~ pro_party:C(topic_id) + C(topic_id) + C(round_number) + C(code)', data=d)
    return m.fit(cov_type='cluster', cov_kwds={'groups': d['code']})


def extract_topic_effects(res):
    topics_map = {1:'Climate',7:'Mobility',5:'Race',2:'Refugees',
                  3:'Obama crime',6:'Gender',4:'Gun laws',8:'Media',9:'Party score'}
    rows, ci = [], res.conf_int()
    for tid, name in topics_map.items():
        matches = [v for v in res.params.index if f'C(topic_id)[{tid}]' in v]
        if len(matches) == 0: continue
        v = matches[0]
        rows.append({'topic': name, 'coef': res.params[v],
                     'ci_lo': ci.loc[v, 0], 'ci_hi': ci.loc[v, 1]})
    return pd.DataFrame(rows)


def plot_figure4(tr):
    fig, ax = plt.subplots(figsize=(8, 6))
    ys = np.arange(len(tr))
    ax.scatter(tr['coef'], ys, color='black', s=40)
    ax.hlines(ys, tr['ci_lo'], tr['ci_hi'], color='black')
    ax.axvline(0, color='red', linestyle='--')
    ax.set_yticks(ys); ax.set_yticklabels(tr['topic'])
    ax.set_xlabel('Effect of Pro-Party News on Pr(True) by Topic')
    fig.tight_layout(); fig.savefig(FIGURES_DIR / 'figure4_replication.png', dpi=150); plt.show()


res = fit_topic_model(samples['main'])
topic_res = extract_topic_effects(res)
plot_figure4(topic_res)
print(topic_res.round(3))

### Table 2: Main Regression — Motivated Reasoning and Perceived News Truthfulness
6 OLS specifications predicting prob_true (stated Pr(True News)).
Key coefficient: beta on pro_party (vs anti_party baseline).
All regressions include round FE. Columns 2-6 add subject FE.
Cluster SEs at subject (code) level throughout.
Mean of dependent variable = 0.574 (55/10 scale normalized to 0-1).

In [ ]:
def run_col1(df):
    """Column 1: Demographic controls, no subject FE."""
    formula = ('prob_true ~ pro_party + C(round_number) + C(topic_id) '
               '+ male + white + log_inc + edu + religious_group + red_state')
    res = smf.ols(formula, data=df).fit(
        cov_type='cluster', cov_kwds={'groups': df['code']})
    return res

def run_col2(df):
    """Column 2: Subject FE + question FE + round FE (within estimator)."""
    formula = 'prob_true ~ pro_party + C(round_number) + C(topic_id) + C(code)'
    res = smf.ols(formula, data=df).fit(
        cov_type='cluster', cov_kwds={'groups': df['code']})
    return res

col1 = run_col1(samples['main'])
col2 = run_col2(samples['main'])
print("Col 1 pro_party coef:", round(col1.params['pro_party'], 3),
      "SE:", round(col1.bse['pro_party'], 3))
print("Col 2 pro_party coef:", round(col2.params['pro_party'], 3),
      "SE:", round(col2.bse['pro_party'], 3))

### Table 2 Columns 3-6

---


Col 3: Interact pro_party with partisanship (abs_net_party). Expect positive interaction (around 0.099).
Col 4: Add both pro_party AND anti_party dummies (relative to neutral baseline).
Col 5: Replace pro_party with true_news indicator (expect negative coef, around -0.059).
Col 6: Include both pro_party and true_news in same model (around 0.077 and -0.034).


In [ ]:
def run_col3(df):
    """Column 3: Interaction with partisanship strength."""
    df = df.copy(); df['part_x_pro'] = df['abs_net_party'] * df['pro_party']
    formula = 'prob_true ~ pro_party + part_x_pro + C(round_number) + C(topic_id) + C(code)'
    return smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['code']})

def run_col5(df):
    """Column 5: True News dummy (instead of pro_party)."""
    formula = 'prob_true ~ true_news + C(round_number) + C(topic_id) + C(code)'
    return smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['code']})

def run_col6(df):
    """Column 6: Both pro_party and true_news in same model."""
    formula = 'prob_true ~ pro_party + true_news + C(round_number) + C(topic_id) + C(code)'
    return smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['code']})

col3 = run_col3(samples['main'])
col5 = run_col5(samples['main'])
col6 = run_col6(samples['main'])
print("Col 3 interaction coef:", round(col3.params['part_x_pro'], 3))
print("Col 5 true_news coef:", round(col5.params['true_news'], 3))
print("Col 6 pro_party:", round(col6.params['pro_party'], 3),
      "| true_news:", round(col6.params['true_news'], 3))

In [ ]:
def format_coef(model, var, decimals=3):
    """Format coefficient and SE for display."""
    if var not in model.params: return '-', '-'
    coef = round(model.params[var], decimals)
    se = round(model.bse[var], decimals)
    stars = '***' if model.pvalues[var] < 0.001 else ''
    return f"{coef}{stars}", f"({se})"

rows = {
    'Pro-Party': [col1, col2, col3, None, None, col6],
    'True News': [None, None, None, None, col5, col6],
}
for label, models in rows.items():
    var = 'pro_party' if label == 'Pro-Party' else 'true_news'
    vals = [format_coef(m, var)[0] if m is not None else '-' for m in models]
    print(f"{label:20}", " | ".join([f"{v:>10}" for v in vals]))

### Table 3: Motivated Reasoning and Second Guesses
Restrict sample to subjects in the second-guess group (your_reguess is not missing).
Dependent variable = change_guess_message (1 if guess moved in direction of news,
-1 if opposite, 0 if unchanged). This validates that subjects genuinely believe
Pro-Party news, not just express preferences through their veracity assessments.


In [ ]:
def run_table3_col1(df_sg):
    """Table 3 Col 1: Pro-Party effect on guess updating."""
    formula = ('change_guess_message ~ pro_party '
               '+ C(round_number) + C(topic_id) + C(code)')
    return smf.ols(formula, data=df_sg).fit(
        cov_type='cluster', cov_kwds={'groups': df_sg['code']})

def run_table3_col4(df_sg):
    """Table 3 Col 4: Add prob_true (assessment) as control."""
    formula = ('change_guess_message ~ pro_party + prob_true '
               '+ C(round_number) + C(topic_id) + C(code)')
    return smf.ols(formula, data=df_sg).fit(
        cov_type='cluster', cov_kwds={'groups': df_sg['code']})

sg = samples['second_guess'].copy()
sg = sg[sg['pro_party'].isin([0,1]) & sg['anti_party'].isin([0,1])]
sg = sg[(sg['pro_party']+sg['anti_party'])==1]
t3c1 = run_table3_col1(sg)
t3c4 = run_table3_col4(sg)
print("Col 1 pro_party:", round(t3c1.params['pro_party'], 3),
      "| Col 4 pro_party (controlling for assessment):", round(t3c4.params['pro_party'], 3))
print("Col 4 prob_true coef:", round(t3c4.params['prob_true'], 3))

### Hypothesis 4: Overprecision and Partisanship
overprecision variable = 1 if the correct answer falls outside subject's IQR.
On politicized topics, expect coverage < 50% (especially for partisans).
On neutral/random-number topics, expect coverage near 50% (or slight underprecision).

In [ ]:
def overprecision_stats(df):
    """Compute coverage rates and test vs 50% for politicized topics."""
    pol = df[(df['politicized_news']==1) & df['overprecision'].notna()]
    overall = 1 - pol['overprecision'].mean()
    partisan_cov = 1 - pol[pol['partisan']==1]['overprecision'].mean()
    moderate_cov = 1 - pol[pol['moderate']==1]['overprecision'].mean()
    t_overall = stats.ttest_1samp(pol['overprecision'], 0.5)
    print(f"Politicized coverage: {overall:.3f} (target: 0.466)")
    print(f"Partisans coverage: {partisan_cov:.3f} (target: 0.442)")
    print(f"Moderates coverage: {moderate_cov:.3f} (target: 0.488)")
    print(f"p-value vs 0.5: {t_overall.pvalue:.4f}")

overprecision_stats(df[df['net_party'] != 0])

### Robustness Check 1: Unskewed Priors (Table 6)
Interact pro_party with unskewed_prior (=1 if guess is exactly midway in IQR).
If skewness drives results, the interaction should be large. The paper finds it is not.

### Robustness Check 2: Prior News Direction (Table 7)
Add count_prev_net (cumulative net pro_party minus anti_party news in prior rounds).
If subjects update across rounds, this should predict current assessments. It does not.


In [ ]:
def run_unskewed_robustness(df):
    """Table 6: Interaction with unskewed prior dummy."""
    d = df[df['unskewed_prior'].notna()].copy()
    d['usk_x_pro'] = d['unskewed_prior'] * d['pro_party']
    formula = ('prob_true ~ pro_party + unskewed_prior + usk_x_pro '
               '+ C(round_number) + C(topic_id) + C(code)')
    res = smf.ols(formula, data=d).fit(
        cov_type='cluster', cov_kwds={'groups': d['code']})
    print("Unskewed x Pro-Party interaction:", round(res.params['usk_x_pro'], 3),
          "(target ~0.016, n.s.)")
    return res

run_unskewed_robustness(samples['main'])

In [ ]:
def run_prior_news_robustness(df):
    """Table 7: Add previous Pro/Anti-Party news count as control."""
    formula = ('prob_true ~ pro_party + count_prev_net '
               '+ C(round_number) + C(topic_id) + C(code)')
    res = smf.ols(formula, data=df).fit(
        cov_type='cluster', cov_kwds={'groups': df['code']})
    print("count_prev_net coef:", round(res.params['count_prev_net'], 4),
          "(target ~-0.001, n.s.)")
    print("pro_party coef unchanged:", round(res.params['pro_party'], 3))
    return res

run_prior_news_robustness(samples['main'])

## Appendix: AI Tool Disclosure

**Tools used**: Claude (Anthropic, claude.ai) via Claude.ai web interface.

**Tasks performed**:
- Translation of Stata .do file logic into Python (variable naming conventions,
  collaps → groupby, regress → statsmodels/linearmodels, cdfplot → matplotlib ECDF)
- Identifying the correct sample filters (net_party != 0, pro_party + anti_party == 1)
  from the Stata code and the paper's text
- Suggesting appropriate Python equivalents for Stata's cluster SE option
- Reviewing modular code structure to comply with the 10-line function limit

**Verification methods**:
- All coefficients were manually checked against paper Tables 2 and 3
- Figure shapes were compared visually against Figures 1-4 in the PDF
- Sample N counts were verified to match paper text (N=7,902 for main analysis)
- Every AI-suggested code cell was re-read, understood, and tested independently

**agent.md / skill.md**: No agent.md or skill.md files were used.